# PRE-WS7 — full game log viewer

Offline-first: every cell below reads a **saved** `game.jsonl`. Nothing here calls an LLM.

- Default log: newest `runs/pre_ws7/*/game.jsonl`, fallback to newest `smoke.jsonl`, fallback to `runs/pre_ws7/fake_gate_seed101/game.jsonl`.
- Run from the repo root so imports resolve. In VS Code pick the `.venv` interpreter, or `uv run --with jupyterlab jupyter lab`.
- The only live-LLM cell is the last one, clearly marked, **disabled by default**. The gate (Cell: verdict) only reads the saved file.
- Token accounting (Cell 5) is the four numbers you asked for: players input/output sums + referee input/output sums.


In [27]:
import os
from pathlib import Path

REPO = Path.cwd()
while REPO.name != "llm-3v3-basketball-agents" and REPO.parent != REPO:
    REPO = REPO.parent
os.chdir(REPO)

env_path = REPO / ".env"
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            name, _, value = line.partition("=")
            os.environ.setdefault(name.strip(), value.strip())

key = os.environ.get("OPENROUTER_API_KEY", "")
print("OPENROUTER_API_KEY:", "present" if key else "MISSING (offline cells still work; live cell will fail)")
print("cwd:", Path.cwd())

OPENROUTER_API_KEY: present
cwd: c:\Users\hongy\OneDrive\Documents\Desktop\llm-3v3-basketball-agents


In [28]:
# ── MODEL KNOBS (edit me — this is where your free models go) ──
# Option A (recommended, keeps `git diff -- configs/` empty for the gate):
#   leave configs/llm.yaml alone; set the slugs here. They are applied
#   in-memory via model_copy(update=...) exactly like scripts/pre_ws7_full_game.py.
# Option B: edit configs/llm.yaml directly (players.model line 10,
#   referee.model line 26). Then the gate manifest must record configs_clean=false.
PLAYER_MODEL = "minimax/minimax-m3.1-flash"   # paid fallback; replace with a live :free slug
REFEREE_MODEL = "openai/gpt-oss-120b"          # replace with a live :free slug if you prefer
APPLY_OVERRIDE = False   # True = use the two constants above in-memory; False = use file verbatim
GAME_JSON = "latest"   # "latest" auto-picks newest game.jsonl (see next cell)

from bball3x3.config import load_llm_config
FILE_CFG = load_llm_config()
print(f"FILE  players={FILE_CFG.players.model}  referee={FILE_CFG.referee.model} {FILE_CFG.referee.enabled}/{FILE_CFG.referee.backend}")
print(f"FILE  temperature={FILE_CFG.referee.temperature} top_k={FILE_CFG.referee.top_k_rules}")
assert FILE_CFG.referee.temperature == 0.0 and FILE_CFG.referee.top_k_rules == 3, "referee temp/top_k drifted"
EFF_PLAYER = PLAYER_MODEL if APPLY_OVERRIDE else FILE_CFG.players.model
EFF_REFEREE = REFEREE_MODEL if APPLY_OVERRIDE else FILE_CFG.referee.model
print(f"EFFECTIVE players={EFF_PLAYER}")
print(f"EFFECTIVE referee={EFF_REFEREE} enabled=True backend=llm (in-memory only)")
print("configs/llm.yaml untouched by this notebook (it only reads).")

FILE  players=nvidia/nemotron-3-super-120b-a12b:free  referee=nvidia/nemotron-3-ultra-550b-a55b:free True/llm
FILE  temperature=0.0 top_k=3
EFFECTIVE players=nvidia/nemotron-3-super-120b-a12b:free
EFFECTIVE referee=nvidia/nemotron-3-ultra-550b-a55b:free enabled=True backend=llm (in-memory only)
configs/llm.yaml untouched by this notebook (it only reads).


In [29]:
import glob, json
from pathlib import Path

def _pick_log(spec="latest"):
    if spec != "latest":
        return Path(spec)
    for pat in ["runs/pre_ws7/*/game.jsonl", "runs/pre_ws7/*/smoke.jsonl", "runs/pre_ws7_smoke/*.jsonl"]:
        hits = sorted(glob.glob(pat))
        if hits:
            return Path(hits[-1])
    return Path("runs/pre_ws7/fake_gate_seed101/game.jsonl")

LOG = _pick_log(GAME_JSON)
LOG = Path("runs/pre_ws7/20261023_021223_seed_101/notebook_live.jsonl")
print("LOG =", LOG, "exists:", LOG.exists())

from bball3x3.analysis import read_game_log, render_possession_transcript
from bball3x3.analysis.possessions import validate_possessions
from bball3x3.analysis.metrics import compute_game_metrics

game = read_game_log(LOG)
poss = validate_possessions(game)
metrics = compute_game_metrics(game)
print("strict read OK; possessions closed OK; metrics OK")
print("record types:", {k: len(v) for k, v in game.by_type.items()})

LOG = runs\pre_ws7\20261023_021223_seed_101\notebook_live.jsonl exists: True
strict read OK; possessions closed OK; metrics OK
record types: {'game_start': 1, 'decision': 60, 'message': 84, 'captain_call': 20, 'memory_proposal': 56, 'memory_write': 56, 'cs_proposal': 52, 'cs_write': 52, 'referee': 10, 'resolution': 10, 'state': 10, 'possession_end': 5}


In [30]:
gs = game.by_type["game_start"][0]
ends = game.by_type.get("game_end", [])
end = ends[-1] if ends else None
res = game.by_type.get("resolution", [])
print(f"game={gs.game_id}  seed={gs.seed}  experiment={gs.experiment_id}")
print(f"events={len(res)}  score={end.final_score if end else '?'}  end={end.end_reason if end else 'UNFINISHED'}  ot={end.ot_flag if end else '?'}")
print(f"cost_total=${(end.total_cost_usd if end else 0):.4f}")
decs = game.by_type.get("decision", [])
pf = sum(1 for d in decs if not d.parse_ok) / max(len(decs), 1)
iar = sum(1 for d in decs if d.invalid_reason is not None) / max(len(decs), 1)
print(f"decisions={len(decs)}  parse_fail={pf:.2%} (target <2%)  IAR={iar:.2%} (target <5%)  SIR violations={len(game.by_type.get('invariant_violation', []))} (must be 0)")
print(f"config_hash={gs.config_hash[:12]}...  model_slug={gs.model_slug}")

game=pre_ws7_notebook_live  seed=101  experiment=pre_ws7
events=10  score=?  end=UNFINISHED  ot=?
cost_total=$0.0000
decisions=60  parse_fail=0.00% (target <2%)  IAR=0.00% (target <5%)  SIR violations=0 (must be 0)
config_hash=d556eeb268af...  model_slug=nvidia/nemotron-3.5-lightning:free


In [31]:
import pandas as pd
from IPython.display import display

# DecisionRecord.tokens is a dict {prompt, completion, total}; RefereeRecord is flat fields.
decs = game.by_type.get("decision", [])
refs = game.by_type.get("referee", [])
p_in  = sum((d.tokens or {}).get("prompt", 0) for d in decs)
p_out = sum((d.tokens or {}).get("completion", 0) for d in decs)
p_cost = sum(d.cost_usd or 0 for d in decs)
p_lat  = sum(d.latency_ms or 0 for d in decs)
r_in  = sum(r.prompt_tokens or 0 for r in refs)
r_out = sum(r.completion_tokens or 0 for r in refs)
r_cost = sum(r.cost_usd or 0 for r in refs)
r_lat  = sum(r.latency_ms or 0 for r in refs)
tot = pd.DataFrame([
    {"scope": "players (6 agents summed)", "input_tokens": p_in, "output_tokens": p_out, "total_tokens": p_in + p_out, "cost_usd": round(p_cost, 4), "latency_ms": int(p_lat)},
    {"scope": "referee (judge summed)",      "input_tokens": r_in, "output_tokens": r_out, "total_tokens": r_in + r_out, "cost_usd": round(r_cost, 4), "latency_ms": int(r_lat)},
    {"scope": "GRAND TOTAL",                 "input_tokens": p_in + r_in, "output_tokens": p_out + r_out, "total_tokens": p_in + p_out + r_in + r_out, "cost_usd": round(p_cost + r_cost, 4), "latency_ms": int(p_lat + r_lat)},
])
display(tot)
print(f"players:  in={p_in}  out={p_out}")
print(f"referee:  in={r_in}  out={r_out}")
if p_in + p_out + r_in + r_out == 0:
    print("NOTE: all token sums are 0 — this log came from cache or FakeLLM; check cost_usd before trusting it.")

# per-player split (spots a runaway agent)
per = {}
for d in decs:
    a = per.setdefault(d.player, {"player": d.player, "n": 0, "in": 0, "out": 0, "cost": 0.0})
    a["n"] += 1; a["in"] += (d.tokens or {}).get("prompt", 0); a["out"] += (d.tokens or {}).get("completion", 0); a["cost"] += d.cost_usd or 0
display(pd.DataFrame(sorted(per.values(), key=lambda r: r["player"])))
# per-event referee tokens
display(pd.DataFrame([{"event": r.event_idx, "in": r.prompt_tokens, "out": r.completion_tokens, "cost": round(r.cost_usd or 0, 5), "call": r.decision.call} for r in refs]).head(30))
print(f"… showing 30/{len(refs)} referee-token rows (full table is in the audit cell)")

,scope,input_tokens,output_tokens,total_tokens,cost_usd,latency_ms
0,players (6 agents summed),156731,307322,464053,0,4034615
1,referee (judge summed),18014,11714,29728,0,353382
2,GRAND TOTAL,174745,319036,493781,0,4387997


players:  in=156731  out=307322
referee:  in=18014  out=11714


,player,n,in,out,cost
0,A1,10,29031,51214,0.0
1,A2,10,26233,50288,0.0
2,A3,10,26186,49535,0.0
3,B1,10,26709,64048,0.0
4,B2,10,24250,31520,0.0
5,B3,10,24322,60717,0.0


,event,in,out,cost,call
0,1,1383,402,0,NONE
1,2,1861,1034,0,NONE
2,3,1860,2000,0,NONE
3,4,1806,2000,0,NONE
4,5,1805,996,0,NONE
5,6,1804,801,0,NONE
6,7,1911,1513,0,NONE
7,8,1888,341,0,NONE
8,9,1809,1603,0,NONE
9,10,1887,1024,0,NONE


… showing 30/10 referee-token rows (full table is in the audit cell)


In [32]:
import pandas as pd
from IPython.display import display

# EVERY referee call — nothing sampled away
refs = game.by_type.get("referee", [])
rows = []
for r in refs:
    d = r.decision
    rows.append({
        "event": r.event_idx,
        "call": d.call,
        "offender": d.offender,
        "fouled": d.fouled_player,
        "foul_type": d.foul_type,
        "violation": d.violation_type,
        "chunks": ",".join(r.retrieved_chunk_ids or []),
        "cited": ",".join(d.rule_ids_cited or []),
        "reject": r.rejection_reason,
        "parse_fail": r.failure_code,
        "model": (r.model or "")[:28],
        "tok_in": r.prompt_tokens,
        "tok_out": r.completion_tokens,
        "lat_ms": int(r.latency_ms or 0),
    })
df = pd.DataFrame(rows)
pd.set_option("display.max_rows", 300)
display(df)
print(f"n={len(df)}  NONE={(df['call']=='NONE').sum()}  non-NONE={(df['call']!='NONE').sum()}  parse_fail={(df['parse_fail'].notna()).sum()}  rejected={(df['reject'].notna()).sum()}")
bad = df[(df["call"] != "NONE") & (df["cited"] == "")]
print(f"non-NONE missing citations: {len(bad)} (must be 0)" + ("  EVENTS: " + ",".join(map(str, bad['event'].tolist())) if len(bad) else ""))

,event,call,offender,fouled,foul_type,violation,chunks,cited,reject,parse_fail,model,tok_in,tok_out,lat_ms
0,1,NONE,None,None,None,None,"R29.1.1,R29.1.3,R29.1.2",,None,REF_PARSE_FAIL,nvidia/nemotron-3-ultra-550b,1383,402,12558
1,2,NONE,None,None,None,None,"R29.1.1,R29.1.3,R29.1.2",,None,NaN,nvidia/nemotron-3-ultra-550b,1861,1034,31806
2,3,NONE,None,None,None,None,"R29.1.1,R29.1.3,R29.1.2",,None,REF_PARSE_FAIL,nvidia/nemotron-3-ultra-550b,1860,2000,64295
3,4,NONE,None,None,None,None,"R29.1.1,R29.1.3,R29.1.2",,None,REF_PARSE_FAIL,nvidia/nemotron-3-ultra-550b,1806,2000,41744
4,5,NONE,None,None,None,None,"R29.1.1,R29.1.3,R29.1.2",,None,NaN,nvidia/nemotron-3-ultra-550b,1805,996,23818
5,6,NONE,None,None,None,None,"R29.1.1,R29.1.3,R29.1.2",,None,NaN,nvidia/nemotron-3-ultra-550b,1804,801,15039
6,7,NONE,None,None,None,None,"R31.3.2,R34.2.2,R31.1.1",,None,NaN,nvidia/nemotron-3-ultra-550b,1911,1513,32654
7,8,NONE,None,None,None,None,"R29.1.1,R29.1.3,R29.1.2",,None,NaN,nvidia/nemotron-3-ultra-550b,1888,341,36193
8,9,NONE,None,None,None,None,"R29.1.1,R29.1.3,R29.1.2",,None,NaN,nvidia/nemotron-3-ultra-550b,1809,1603,72935
9,10,NONE,None,None,None,None,"R29.1.1,R29.1.3,R29.1.2",,None,NaN,nvidia/nemotron-3-ultra-550b,1887,1024,22336


n=10  NONE=10  non-NONE=0  parse_fail=3  rejected=0
non-NONE missing citations: 0 (must be 0)


In [33]:
import json

def _compact(rec: dict) -> str:
    kind = rec.get("record_type", "?")
    e = rec.get("event_idx")
    pre = f"E{e} " if e is not None else ""
    if kind == "game_start":
        return f"[game_start] model={rec.get('model_slug')} seed={rec.get('seed')}"
    if kind == "game_end":
        return f"[game_end] {json.dumps(rec.get('totals') or rec.get('final_score') or {}, default=str)[:200]}"
    if kind == "decision":
        tgt = f" {rec['executed_target']}" if rec.get("executed_target") else ""
        tok = (rec.get("tokens") or {}).get("total")
        return f"[decision] {pre}{rec.get('player')} -> {rec.get('executed_action')}{tgt} parse_ok={rec.get('parse_ok')} tok={tok}"
    if kind == "message":
        return f"[message] {pre}{rec.get('sender')} ({rec.get('kind')}): {(rec.get('text') or '')[:160]}"
    if kind == "captain_call":
        t = rec.get("tactic")
        t = t.get("text") if isinstance(t, dict) else getattr(t, "text", t)
        return f"[captain_call] {pre}team={rec.get('team')} maintain={rec.get('maintain')} tactic={str(t or '')[:120]!r}"
    if kind == "referee":
        dec = rec.get("decision") or {}
        return (f"[referee] {pre}{dec.get('call')} chunks={rec.get('retrieved_chunk_ids')} "
                f"cited={dec.get('rule_ids_cited')} fail={rec.get('failure_code')} tok={rec.get('prompt_tokens')}/{rec.get('completion_tokens')}")
    if kind == "resolution":
        return f"[resolution] {pre}{json.dumps(rec.get('actions'), default=str)} scored={bool((rec.get('scoring_events') or []))}"
    if kind == "state":
        st = rec.get("state") or {}
        return f"[state] {pre}score={st.get('score')} clock={st.get('game_clock_s')} shot={st.get('shot_clock_s')} offense={st.get('offense')}"
    if kind == "possession_end":
        return f"[possession_end] {rec.get('possession_id')} team={rec.get('team')} pts={rec.get('points')} reason={rec.get('end_reason')}"
    if kind == "invariant_violation":
        return f"[VIOLATION] {pre}{rec.get('constraint_name')}: {str(rec.get('details'))[:160]}"
    return f"[{kind}] {pre}{json.dumps({k: v for k, v in rec.items() if k not in ('record_type','event_idx') and v is not None}, default=str)[:220]}"

SHOW_EVERY_EVENT = True   # False = first 3 events only (quick skim)
raw = [json.loads(l) for l in Path(LOG).read_text(encoding="utf-8").splitlines() if l.strip()]
if SHOW_EVERY_EVENT:
    for r in raw:
        print(_compact(r))
else:
    seen, shown = set(), 0
    for r in raw:
        if r.get("event_idx") in (1, 2, 3) or r.get("record_type") in ("game_start", "game_end"):
            print(_compact(r)); shown += 1
    print(f"… {shown}/{len(raw)} lines shown (set SHOW_EVERY_EVENT=True for all)")
print(f"total lines: {len(raw)}")

[game_start] model=nvidia/nemotron-3.5-lightning:free seed=101
[decision] E1 A1 -> DRIVE MIDDLE parse_ok=True tok=8365
[decision] E1 A2 -> CUT PAINT parse_ok=True tok=2812
[decision] E1 A3 -> CUT PAINT parse_ok=True tok=3515
[decision] E1 B1 -> CONTEST A1 parse_ok=True tok=6544
[decision] E1 B2 -> DENY A2 parse_ok=True tok=3631
[decision] E1 B3 -> CONTEST A1 parse_ok=True tok=5657
[message] E1 A1 (action): cut to the rim
[captain_call] E1 team=A maintain=False tactic='drive middle, A3 cut to rim'
[message] E1 A1 (tactic): drive middle, A3 cut to rim
[message] E1 A2 (info): cutting to the paint for a feed from A1
[message] E1 A3 (action): looking for pass
[message] E1 B1 (info): I've got A1, contesting the shot
[captain_call] E1 team=B maintain=False tactic='tight on-ball defense'
[message] E1 B1 (tactic): tight on-ball defense
[message] E1 B2 (info): [defend A2 tight] deny the pass, A2 is foul prone [/defend]
[message] E1 B3 (info): contest the shot
[memory_proposal] E1 {"team": "A", "

In [34]:
import pandas as pd
from IPython.display import display

# EVERY decision (one row per player per event)
decs = game.by_type.get("decision", [])
display(pd.DataFrame([{
    "event": d.event_idx, "player": d.player,
    "executed": d.executed_action, "target": d.executed_target,
    "parse_ok": d.parse_ok, "invalid": d.invalid_reason,
    "tok_in": (d.tokens or {}).get("prompt"), "tok_out": (d.tokens or {}).get("completion"),
    "lat_ms": int(d.latency_ms or 0),
} for d in decs]))
print(f"decisions={len(decs)}")

,event,player,executed,target,parse_ok,invalid,tok_in,tok_out,lat_ms
0,1,A1,DRIVE,MIDDLE,True,None,2261,6104,80257
1,1,A2,CUT,PAINT,True,None,2012,800,11561
2,1,A3,CUT,PAINT,True,None,2011,1504,19273
3,1,B1,CONTEST,A1,True,None,2337,4207,38305
4,1,B2,DENY,A2,True,None,2068,1563,18025
5,1,B3,CONTEST,A1,True,None,2067,3590,34665
6,2,A1,DENY,B1,True,None,2476,4083,42469
7,2,A2,DENY,B2,True,None,2214,5120,60929
8,2,A3,DENY,B3,True,None,2207,6241,100412
9,2,B1,CLEAR,NaN,True,None,2393,9746,150755


decisions=60


In [35]:
# Transcripts: opening possession, first non-NONE referee call, closing possession.
refs = game.by_type.get("referee", [])
res = game.by_type.get("resolution", [])
first_call = next((r.event_idx for r in refs if r.decision.call != "NONE"), None)
print("first non-NONE referee event:", first_call)
for pid in [res[0].possession_before.possession_id if res else "p0001"]:
    print(render_possession_transcript(game, pid))
    print("=" * 72)

first non-NONE referee event: None
Game pre_ws7_notebook_live | possession p0001 | team A
Game log: unfinished | possession: closed | events 1–1
Conditions: A=A_full, B=A_full
Possession points: A=0, B=0 | end reason: STRIP
Boundary events may be shared with adjacent possessions.
Decisions belong to possession_before; scoring entries retain their possession IDs.
Recorded observations are authoritative; missing observation content is not inferred.

Event 1
Decision A1 (team A)
  Observation:
    [OWN STATE]
    Player: A1
    Team: A
    Role: BALL_HANDLER
    Position: (7, 9)
    Has Ball: true
    Fatigue: 0.0/10
    
    [TEAMMATES]
    A2: pos=(3, 7), movement=HOLD, speed=NONE
    A3: pos=(11, 7), movement=HOLD, speed=NONE
    
    [OPPONENTS]
    B1: pos=(7, 6), movement=HOLD, speed=NONE
    B2: pos=(3, 5), movement=HOLD, speed=NONE
    B3: pos=(11, 5), movement=HOLD, speed=NONE
    
    [DEFENSIVE MATCHUPS]
    A1 -> B1
    A2 -> B2
    A3 -> B3
    B1 -> A1
    B2 -> A2
    B3 ->

In [36]:
import subprocess
refs = game.by_type.get("referee", [])
res = game.by_type.get("resolution", [])
ends = game.by_type.get("game_end", [])
end = ends[-1] if ends else None
diff = subprocess.run(["git", "diff", "--", "configs/"], capture_output=True, text=True).stdout.strip()
checks = {
    "terminated": end is not None and end.end_reason in ("WIN_SCORE", "CLOCK_EXPIRED", "OT_WIN"),
    "one referee per event": len(refs) == len(res) > 0,
    "non-NONE all cited": all((d.rule_ids_cited for d in (r.decision for r in refs) if d.call != "NONE")),
    "SIR == 0": len(game.by_type.get("invariant_violation", [])) == 0,
    "configs untouched": diff == "",
}
for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
print("PRE_WS7 GATE:", "PASS" if all(checks.values()) else "FAIL")

FAIL  terminated
PASS  one referee per event
PASS  non-NONE all cited
PASS  SIR == 0
FAIL  configs untouched
PRE_WS7 GATE: FAIL


In [26]:
# Set RUN_LIVE = True to play a fresh game with models from Cell 2.
RUN_LIVE = True
LIVE_EVENTS = 10

if RUN_LIVE:
    import asyncio, datetime
    from bball3x3.config import load_llm_config as _load
    from bball3x3.llm.client import OpenRouterClient
    from bball3x3.runner.llm_game import run_llm_game as _run
    _cfg = _load()
    _live = _cfg.model_copy(update={
        "players": _cfg.players.model_copy(update={"model": EFF_PLAYER}),
        "referee": _cfg.referee.model_copy(update={"enabled": True, "backend": "llm", "model": EFF_REFEREE}),
    })
    _stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    _out = f"runs/pre_ws7/{_stamp}_seed_101/notebook_live.jsonl"
    _pb = OpenRouterClient(_live)
    try:
        _res = await _run(llm=_pb, path=_out, seed=101, llm_config=_live, max_events=LIVE_EVENTS,
                          experiment_id="pre_ws7", game_id="pre_ws7_notebook_live", replicate_id="seed_101")
    finally:
        try: await _pb.aclose()
        except Exception: pass
    print(_res)
    print("re-point GAME_JSON at", _out, "and Run All again for the offline view")
else:
    print("live run skipped (RUN_LIVE=False) — gate cells above are offline-only")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1587.16it/s]


{'path': 'runs\\pre_ws7\\20261023_021223_seed_101\\notebook_live.jsonl', 'events': 10, 'score': {'A': 0, 'B': 2}, 'game_complete': False, 'end_reason': None, 'decisions_by_team': {'A': 30, 'B': 30}, 'possessions_by_team': {'A': 3, 'B': 2}, 'parse_failure_rate': 0.0, 'iar': 0.0, 'total_cost_usd': 0.0}
re-point GAME_JSON at runs/pre_ws7/20261023_021223_seed_101/notebook_live.jsonl and Run All again for the offline view
